CV
-
divisione frasi in blocchi
-
sentence transformer multilingue
-
embedding di ogni blocco
- 
confronto con Job Description e compentenze.


CV italiano
CV inglese
CV francese                 stesso modello multilingue
CV tedesco
CV spagnolo 

In [ ]:
#python -m spacy download en_core_web_sm
# python -m pip install sentence-transformers
import os, re, fitz, torch
from sentence_transformers import SentenceTransformer, util
from pathlib import Path

# ==================================================================================
# PARAMETRI DI ANALISI SEMANTICA
# ==================================================================================
# Definiamo i concetti fondamentali che ricerchiamo nei candidati.
# Nota: Grazie al Deep Matching, il sistema troverà questi concetti anche se espressi 
# con parole diverse (es: "Coding" attiverà "Programming").

# PROFILO IDEALE
TARGET_CONCEPTS = ["python", "programming", "data science", "ai", "statistics", "mathematics", "sql"]

# Soglia iniziale per considerare una competenza semanticamente compatibile.
# ATTENZIONE: 0.55 NON significa "55% di certezza".
# È una soglia euristica da calibrare su CV reali.
SKILL_THRESHOLD = 0.55

# Modello BERT: trasforma parole e frasi in vettori in uno spazio multidimensionale.
MODEL_NAME = ("sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2")

# ==================================================================================
# CLASSE CORE: NeuralRecruiter (racchiude tutta la lofica)
# Un sistema di recruiting di nuova generazione che non si limita alle "parole chiave",
# ma comprende il significato dei concetti tecnici espressi nel CV.
# ==================================================================================
class NeuralRecruiter:
    def __init__(self):
        """
        Inizializza il modello semantico e
        pre-calcola gli embedding delle competenze target.
        """

        print("[SISTEMA] Avvio NeuralRecruiter...")

        # ----------------------------------------------------------
        # 1. DEVICE
        # ----------------------------------------------------------

        self.device = ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[SISTEMA] Device utilizzato: {self.device}")

        # ----------------------------------------------------------
        # 2. MODELLO SEMANTICO MULTILINGUE
        # ----------------------------------------------------------

        print(f"[SISTEMA] Caricamento Sentence Transformer multilingue - {MODEL_NAME}")
        self.model = SentenceTransformer(MODEL_NAME,device=self.device)

        
        # ----------------------------------------------------------
        # 3. PRE-CALCOLO EMBEDDING DELLE SKILL TARGET
        # ----------------------------------------------------------

        print(f"[SISTEMA] Indicizzazione vettoriale delle competenze target: {TARGET_CONCEPTS}")
        self.target_vecs = self.get_emb(TARGET_CONCEPTS)


    # ==================================================================================
    # EMBEDDING
    # ==================================================================================

    def get_emb(self, text): #funzione centrale per BERT
        """
        Trasforma una stringa oppure una lista di stringhe
        in sentence embeddings.

        A differenza del vecchio BERT:
        - non facciamo tokenizer manualmente;
        - non prendiamo last_hidden_state;
        - non facciamo noi la media dei token.

        Il modello è già progettato per produrre
        rappresentazioni utili alla semantic similarity.
        """
        return self.model.encode(text,convert_to_tensor=True, normalize_embeddings=True)        

    # ==================================================================================
    # SIMILARITÀ
    # ==================================================================================

    def cosine_sim(self, v1, v2): #confronto di due vettori
        """
        Calcola la cosine similarity tra due embedding.
        -1 = direzioni ooposte
         0  = ortogonali
        +1 = stessa direzione        
        """
        return util.cos_sim(v1, v2)

    # ==================================================================================
    # SUDDIVISIONE DEL CV
    # ==================================================================================

    def split_into_chunks(self, text):
        """
        Divide il CV in frasi / blocchi.

        Non vogliamo trasformare un CV di più pagine in un unico enorme embedding.
        Preferiamo:
        CV
        -> blocco 1
        -> blocco 2
        -> blocco 3
        -> ...
        """

        # Normalizziamo ritorni a capo Windows/Linux.
        text = text.replace("\r", "\n")

        # Divido in questo modo:
        # 1. (?<=[.!?]) : uno o più spazzi dopo . ! ? - le [] indicano 'uno qualsiasi tra' ?<= \s=spazio bianco e + significa uno o più
        # 2. \n+: uno o più ritorni a capo
        pieces = re.split(r'(?<=[.!?])\s+|\n+',text)

        chunks = []
        for piece in pieces:
            # Normalizzazione spazi
            piece = re.sub(r'\s+',' ', piece).strip()
            # Ignoriamo frammenti troppo piccoli (<10)
            if len(piece) >= 10: chunks.append(piece)

        # Nel caso non sia stato possibile individuare nessun blocco.
        if not chunks:
            chunks = [text.strip()]

        return chunks

    # ==================================================================================
    # ESTRAZIONE NOME
    # ==================================================================================

    def extract_name(self, text):
        """
        Estrazione molto semplice del nome.
        Per questo esercizio assumiamo che il nome si trovi all'inizio del CV.
        NON è un NER professionale.
        In produzione potremmo aggiungere un modello NER multilingue.
        """
        # Prendiamo solo la parte iniziale (assumendo che all'inizio di sia il nome, altrimenti fallisce)
        first_part = re.split(r'[\n.]',text.strip(), maxsplit=1)[0]
        # re.split vuol dire dividi il testo ogni volta che trovi il pattern indicato e genera una lista
        # [\n.]  [] vuol dire qualsiasi carattere dentro le parentesi  \n ritorno a capo . punto
        # maxsplit=1 quante divisione fare, in questo caso 1
        # [0] prende solo il primo elemento della lista generata con re.split

        # Cerchiamo 2-4 parole che sembrano un nome proprio
        pattern = (
            r"^("   # ^=inizio stringa
            r"[A-ZÀ-ÖØ-Ý][A-Za-zÀ-ÖØ-öø-ÿ'’-]+" #la prima lettera deve essere maiuscola, poi altre lettere (maiuscole/minuscole)  += uno o più caratteri
            r"(?:\s+"  #\s+ uno o più spazi
            r"[A-ZÀ-ÖØ-Ý][A-Za-zÀ-ÖØ-öø-ÿ'’-]+"  #poi altra parola con la prima lettere maiuscola e le altre indifferente
            r"){1,3}"  #ripeti tutto da 1 a 3 volte
            r")"
        )
        match = re.search(pattern,first_part)
        return (match.group(1) if match else "Sconosciuto")    
    
 
    # ==================================================================================
    # ANALISI CV
    # ==================================================================================

    def analyze(self, text, jd_vec):
        """
        Analisi multicriterio del CV.

        Combina:
        - Regex per dati strutturati
        - Sentence Embeddings
        - Cosine Similarity
        - Regole di scoring definite dal programmatore
        """

        # ------------------------------------------------------------------
        # 1. DATI STRUTTURATI (Regex)
        # ------------------------------------------------------------------

        email = re.search(r'[\w\.-]+@[\w\.-]+\.\w+',text)
        name = self.extract_name(text)

        # ------------------------------------------------------------------
        # 2. DIVISIONE CV IN BLOCCHI
        # ------------------------------------------------------------------

        chunks = self.split_into_chunks(text)
        print(f"    Blocchi semantici trovati: {len(chunks)}")

        # ------------------------------------------------------------------
        # 3. EMBEDDING DEI BLOCCHI
        # ------------------------------------------------------------------

        # IMPORTANTE:
        # facciamo una sola chiamata batch al modello.
        # non fiacciamo una chimata per ogni chunk ma creiamo una lista di chunk e poi chiamata al modello
        #

        chunk_vecs = self.get_emb(chunks)

        # ------------------------------------------------------------------
        # 4. MATCH GLOBALE CON LA JOB DESCRIPTION
        # ------------------------------------------------------------------

        # Confrontiamo OGNI parte del CV con la JD.
        jd_scores = self.cosine_sim(chunk_vecs,jd_vec).squeeze(1)

        # Prendiamo i 3 blocchi più pertinenti.
        # In questo modo formazione, indirizzo, ecc.
        # non abbassano artificialmente il punteggio.
        top_k = min(3,len(jd_scores))
        top_scores = torch.topk(jd_scores,k=top_k).values
        global_score = (top_scores.mean().item())

        # Evitiamo valori fuori dall'intervallo 0-1
        # nel nostro score applicativo.
        global_score = max(0.0,min(global_score, 1.0))


        # ------------------------------------------------------------------
        # 5. DEEP SKILL MATCHING
        # ------------------------------------------------------------------

        # Matrice:
        #
        #                    Python   AI   SQL   ...
        #
        # blocco CV 1          x      x    x
        # blocco CV 2          x      x    x
        # blocco CV 3          x      x    x
        #
        # Ogni x è una cosine similarity.

        skill_matrix = self.cosine_sim(chunk_vecs,self.target_vecs)

        found_skills = []
        skill_details = {}

        # Per ogni competenza target
        for target_index, target in enumerate(TARGET_CONCEPTS):

            # Similarità di tutti i blocchi con questa specifica skill.
            scores = skill_matrix[:,target_index]

            best_score, best_chunk_index = (
                torch.max(scores, dim=0)
            )

            score_value = best_score.item()

            # Registriamo anche l'evidenza:
            # quale frase ha prodotto il match?
            skill_details[target] = {
                "score": score_value,
                "evidence":
                    chunks[
                        best_chunk_index.item()
                    ]
            }

            if score_value >= SKILL_THRESHOLD:
                found_skills.append(target)


        # ------------------------------------------------------------------
        # 6. SCORE FINALE
        # ------------------------------------------------------------------

        # Percentuale delle competenze target trovate.
        skill_coverage = (
            len(found_skills)
            / len(TARGET_CONCEPTS)
        )

        # Manteniamo la filosofia del tuo script:
        #
        # 70% = affinità semantica con la Job Description
        # 30% = copertura delle skill target
        #
        # ATTENZIONE:
        # questi pesi sono BUSINESS RULES.
        # Non sono stati imparati dal modello.

        final_score = (
            global_score * 0.70
            +
            skill_coverage * 0.30
        )


        # ------------------------------------------------------------------
        # 7. RISULTATO STRUTTURATO
        # ------------------------------------------------------------------

        return {
            "name": name,

            "email":
                email.group()
                if email
                else "N.D.",

            "score":
                final_score,

            "semantic_score":
                global_score,

            "skill_coverage":
                skill_coverage,

            "skills":
                found_skills,

            "skill_details":
                skill_details
        }


# ==================================================================================
# UTILITY DI GESTIONE INPUT
# ==================================================================================

def get_text_from_source(source):
    """
    Accetta:
    - testo normale
    - percorso PDF

    Se il PDF contiene testo digitale,
    PyMuPDF lo estrae direttamente.
    """

    source_str = str(source)

    if (
        source_str.lower().endswith(".pdf")
        and os.path.exists(source_str)
    ):

        with fitz.open(source_str) as doc:

            # Manteniamo i ritorni a capo.
            # Nel vecchio script li eliminavamo tutti,
            # ma ci servono per identificare i blocchi del CV.
            pages = [
                page.get_text("text")
                for page in doc
            ]

            return "\n".join(pages)

    return source_str


# ==================================================================================
# ESECUZIONE TEST E CLASSIFICA
# ==================================================================================

if __name__ == "__main__":

    # ------------------------------------------------------------------
    # 1. INIZIALIZZAZIONE
    # ------------------------------------------------------------------

    bot = NeuralRecruiter()


    # ------------------------------------------------------------------
    # 2. CARTELLA DI LAVORO
    # ------------------------------------------------------------------

    # Funziona sia come .py sia in Jupyter.
    try:
        script_dir = os.path.dirname(
            os.path.abspath(__file__)
        )

    except NameError:
        script_dir = os.getcwd()


    # ------------------------------------------------------------------
    # 3. PDF DI TEST
    # ------------------------------------------------------------------

    pdf_path = os.path.join(
        script_dir,
        "cv_test.pdf"
    )

    if not os.path.exists(pdf_path):

        print(
            "[SETUP] Creazione CV "
            "di test in PDF..."
        )

        doc = fitz.open()

        page = doc.new_page()

        page.insert_text(
            (50, 50),
            (
                "Giulia Verdi\n"
                "giulia@mail.com\n"
                "Esperta di Deep Learning.\n"
                "Esperienza in reti neurali e "
                "intelligenza artificiale.\n"
                "Ottima conoscenza del calcolo "
                "matematico e della statistica."
            )
        )

        doc.save(pdf_path)
        doc.close()


    # ------------------------------------------------------------------
    # 4. JOB DESCRIPTION
    # ------------------------------------------------------------------

    # La Job Description è in italiano.
    #
    # Un candidato può però avere il CV
    # in inglese/francese/tedesco/spagnolo.

    query_recruiting = (
        "Cerchiamo un professionista con esperienza "
        "in intelligenza artificiale, programmazione "
        "software e matematica."
    )

    jd_vec = bot.get_emb(
        query_recruiting
    )


    # ------------------------------------------------------------------
    # 5. CANDIDATI
    # ------------------------------------------------------------------

    candidates = [

        # Italiano - non pertinente
        (
            "Mario Rossi. "
            "mario@mail.com. "
            "Esperienza in social media, "
            "marketing e comunicazione."
        ),

        # Inglese - pertinente
        (
            "Luca Bianchi. "
            "luca@tech.it. "
            "Experienced software developer. "
            "Strong knowledge of coding, algorithms "
            "and neural networks. "
            "Worked on artificial intelligence projects."
        ),

        # PDF italiano
        pdf_path
    ]


    # ------------------------------------------------------------------
    # 6. ANALISI
    # ------------------------------------------------------------------

    results = []

    print(
        f"\n{'=' * 20} "
        f"AVVIO ANALISI SEMANTICA "
        f"{'=' * 20}"
    )


    for src in candidates:

        text = get_text_from_source(
            src
        )

        res = bot.analyze(
            text,
            jd_vec
        )

        results.append(res)

        print(
            f"\n -> Analizzato: "
            f"{res['name']:<20}"
        )

        print(
            f"    Skill rilevate: "
            f"{res['skills']}"
        )

        print(
            f"    Similarità JD: "
            f"{res['semantic_score']:.3f}"
        )


    # ------------------------------------------------------------------
    # 7. ORDINAMENTO PER MATCHING SCORE
    # ------------------------------------------------------------------

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )


    # ------------------------------------------------------------------
    # 8. REPORT
    # ------------------------------------------------------------------

    print(
        f"\n{'POS':<5} "
        f"{'MATCH SCORE':<15} "
        f"{'NOME':<20} "
        f"{'COMPETENZE RILEVATE'}"
    )

    print("-" * 100)


    for i, r in enumerate(results, 1):

        concept_str = (
            ", ".join(r["skills"])
            if r["skills"]
            else "Nessuna affinità specifica"
        )

        print(
            f"{i:<5} "
            f"{r['score'] * 100:>6.1f}%        "
            f"{r['name']:<20} "
            f"{concept_str}"
        )


    # ------------------------------------------------------------------
    # 9. EVIDENZE
    # ------------------------------------------------------------------

    print(
        "\n\n"
        + "=" * 30
        + " EVIDENZE SEMANTICHE "
        + "=" * 30
    )

    for r in results:

        print(
            f"\nCANDIDATO: {r['name']}"
        )

        for skill in r["skills"]:

            detail = r[
                "skill_details"
            ][skill]

            print(
                f"\n  {skill}"
            )

            print(
                f"  Similarità: "
                f"{detail['score']:.3f}"
            )

            print(
                f"  Evidenza: "
                f"{detail['evidence']}"
            )